# Bike neighbourhoods — network community detection

Build a weighted undirected station graph from 2025 trips (edge weight = bidirectional trip count), then run Louvain community detection. The question: does the data draw neighbourhoods differently than the City's official boundaries?

Outputs: `data/processed/communities_map.html` (colored ScatterplotLayer), and a summary of community sizes + geography.

Reminder from `03_flow_map.ipynb`: we join by station **id** against the GBFS feed because `End_Station_Name` in the trip data is unreliable.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import duckdb
import pandas as pd
import networkx as nx
import pydeck as pdk
from networkx.algorithms.community import louvain_communities

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")

stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['station_id_int'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
station_lookup = stations.dropna(subset=['station_id_int']).set_index('station_id_int')[['name', 'lat', 'lon']]
len(station_lookup)

1030

## 1. Build edge list

Undirected pairs (a_id, b_id) with a_id < b_id, weight = total trips both directions. Exclude round-trips.

In [2]:
edges = con.execute('''
    SELECT
      LEAST(Start_Station_Id, End_Station_Id) AS a,
      GREATEST(Start_Station_Id, End_Station_Id) AS b,
      COUNT(*) AS weight
    FROM trips
    WHERE Start_Station_Id IS NOT NULL
      AND End_Station_Id IS NOT NULL
      AND Start_Station_Id <> End_Station_Id
    GROUP BY 1, 2
''').fetchdf()
print('Distinct station pairs:', len(edges), 'total trips:', edges['weight'].sum())
edges.head()

Distinct station pairs: 217426 total trips: 7558521


,a,b,weight
0,7118,7299,171
1,7122,7524,189
2,7078,7150,412
3,7100,7114,767
4,7217,7281,593


## 2. Filter to stations present in GBFS

Keep only edges where both endpoints resolve to a current station (have lat/lon). Drops decommissioned stations.

In [3]:
known = set(station_lookup.index.dropna().astype(int))
edges = edges[edges['a'].isin(known) & edges['b'].isin(known)].copy()
print('Edges with both endpoints known:', len(edges), 'trips retained:', edges['weight'].sum())

Edges with both endpoints known: 205040 trips retained: 7126858


## 3. Build graph and run Louvain

Louvain optimizes modularity — it groups nodes such that within-group edges are denser than expected under a random null model. The `resolution` parameter trades off between few-large and many-small communities. Start at 1.0 (default).

In [4]:
G = nx.Graph()
for a, b, w in edges.itertuples(index=False):
    G.add_edge(int(a), int(b), weight=int(w))

print('Nodes:', G.number_of_nodes(), 'edges:', G.number_of_edges())
print('Mean edge weight:', round(edges['weight'].mean(), 1),
      'median:', int(edges['weight'].median()),
      'p95:', int(edges['weight'].quantile(0.95)))

Nodes: 994 edges: 205040
Mean edge weight: 34.8 median: 8 p95: 157


In [5]:
# Deterministic seed so colors are stable across runs.
communities = louvain_communities(G, weight='weight', resolution=1.0, seed=42)
communities.sort(key=len, reverse=True)
print(f'Found {len(communities)} communities')
print('Sizes (top 15):', [len(c) for c in communities[:15]])

Found 6 communities
Sizes (top 15): [250, 215, 170, 126, 122, 111]


In [6]:
# Attach community label to each station.
node_to_community = {n: i for i, comm in enumerate(communities) for n in comm}

station_lookup = station_lookup.copy()
station_lookup['community'] = station_lookup.index.map(node_to_community)

# Drop stations that didn't get a community (no trips in/out, or filtered out).
mapped = station_lookup.dropna(subset=['community']).copy()
mapped['community'] = mapped['community'].astype(int)
print(f'{len(mapped)} of {len(station_lookup)} stations assigned a community')

# Pick a few sample stations from each top community to eyeball the geography.
for i in range(min(8, len(communities))):
    sample = mapped[mapped['community'] == i].nlargest(5, 'lat')['name'].tolist()
    print(f'\ncommunity {i} ({len(communities[i])} stations): {sample}')

994 of 1030 stations assigned a community

community 0 (250 stations): ['Lindylou Rd / Lanyard Rd', ' Silverstone Dr / Stevenson Rd ', 'Martin Grove Rd / Lexington Ave', 'Albion Rd / Kipling Ave (Albion Arena)', 'Thistletown Community Centre']

community 1 (215 stations): ['Starspray Blvd / Lawrence Ave E', 'Livonia Pl / Neilson Rd', 'University of Toronto Scarborough', 'Neilson Rd / Ellesmere Rd', 'Rouge Hill GO Station']

community 2 (170 stations): ['Midland Ave / Finch Recreational Trail', 'Brimwood Blvd / North Scarborough Green Loop', 'Kennedy Rd / Finch Recreational Trail', 'Finch Ave E / Midland Ave', 'Silver Springs Blvd / Birchmount Rd']

community 3 (126 stations): ['University Ave / Gerrard St W - SMART(South)', 'Bay St / Albert St', 'Simcoe St / Michael Sweet Ave', 'Bathurst St / Dundas St W', 'York St / Queen St W (City Hall)']

community 4 (122 stations): ['Spadina Rd / St. Clair Ave W', 'Summerhill Subway Station', 'Spadina Rd / Austin Terrace', 'Avenue Rd / Macpherson 

## 4. Render map

Color each station by its community. Qualitative palette of up to 20; lump tail into grey.

In [7]:
# Tableau-like qualitative palette (RGB), enough for the top communities.
PALETTE = [
    [31, 119, 180], [255, 127, 14], [44, 160, 44], [214, 39, 40], [148, 103, 189],
    [140, 86, 75], [227, 119, 194], [127, 127, 127], [188, 189, 34], [23, 190, 207],
    [174, 199, 232], [255, 187, 120], [152, 223, 138], [255, 152, 150], [197, 176, 213],
    [196, 156, 148], [247, 182, 210], [199, 199, 199], [219, 219, 141], [158, 218, 229],
]
GREY = [180, 180, 180]

def color_for(comm_idx: int) -> list:
    if comm_idx < len(PALETTE):
        return PALETTE[comm_idx] + [220]
    return GREY + [120]

mapped['color'] = mapped['community'].map(color_for)
mapped['radius'] = 60

scatter = pdk.Layer(
    'ScatterplotLayer',
    data=mapped.reset_index(drop=True),
    get_position=['lon', 'lat'],
    get_radius='radius',
    get_fill_color='color',
    pickable=True,
    auto_highlight=True,
)

view = pdk.ViewState(latitude=43.675, longitude=-79.39, zoom=11.5, pitch=0)
deck = pdk.Deck(
    layers=[scatter],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{name}\ncommunity {community}'},
)
out = DATA_PROC / 'communities_map.html'
deck.to_html(str(out), notebook_display=False)
print('Wrote:', out)

Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/communities_map.html


## 5. Community summary

For each community, report centroid, geographic spread, and a few representative stations so we can eyeball what 'neighbourhood' the algorithm is capturing.

In [8]:
summary = (
    mapped.groupby('community')
    .agg(
        n_stations=('name', 'count'),
        lat_centroid=('lat', 'mean'),
        lon_centroid=('lon', 'mean'),
        lat_range=('lat', lambda s: round(s.max() - s.min(), 4)),
        lon_range=('lon', lambda s: round(s.max() - s.min(), 4)),
    )
    .sort_values('n_stations', ascending=False)
)
summary.head(15)

,n_stations,lat_centroid,lon_centroid,lat_range,lon_range
community,,,,,
0,250,43.660883,-79.465875,0.1604,0.1943
1,215,43.706711,-79.291109,0.1340,0.2448
2,170,43.737547,-79.405134,0.1374,0.2622
3,126,43.644236,-79.399039,0.0241,0.0531
4,122,43.664126,-79.391413,0.0356,0.0471
5,111,43.650258,-79.370656,0.0338,0.0879


In [9]:
# Find the 'most central' station in each community (highest weighted degree within-community).
def community_anchor(comm_idx: int) -> str | None:
    nodes = communities[comm_idx]
    subg = G.subgraph(nodes)
    if subg.number_of_nodes() == 0:
        return None
    deg = dict(subg.degree(weight='weight'))
    anchor = max(deg, key=deg.get)
    return station_lookup.loc[anchor, 'name'] if anchor in station_lookup.index else None

for i in range(min(15, len(communities))):
    print(f'community {i:2d} ({len(communities[i])} stations): anchor = {community_anchor(i)}')

community  0 (250 stations): anchor = Bloor St W / Shaw St - SMART
community  1 (215 stations): anchor = Danforth Ave / Gough Ave


community  2 (170 stations): anchor = Yonge St / Orchard View Blvd


community  3 (126 stations): anchor = King St W / Bay St (West Side)


community  4 (122 stations): anchor = Bay St / College St (East Side)
community  5 (111 stations): anchor = York St / Queens Quay W
